In [5]:
from langgraph.types import Command, interrupt

from langgraph.checkpoint.memory import InMemorySaver, MemorySaver
from langchain_core.runnables import RunnableLambda, RunnableBranch, RunnableParallel, branch
from pydantic import BaseModel, Field

from  langchain_openai import ChatOpenAI
import os
from dotenv import load_dotenv
from openai import OpenAI
import httpx

from langchain_core.messages import HumanMessage, SystemMessage, AIMessage, ToolMessage
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from typing import Literal, TypedDict, List, Annotated, Optional
import uuid
import base64
import ssl
from langgraph.graph import StateGraph, START, END, add_messages
from IPython.display import Image,display, Markdown
from langchain.tools import tool
from langchain_community.tools import DuckDuckGoSearchRun
from langchain_community.tools import WikipediaQueryRun
from langchain_community.utilities import WikipediaAPIWrapper



# This function will load all the variables from the .env file and will
# make them available in the os.environ dictionary (env variables)
load_dotenv()
ssl_cert_file=""
client= httpx.Client(
        verify=ssl.create_default_context(cafile=ssl_cert_file)
    )

client_id=os.environ.get('LLM_CLIENT_ID')
client_secret=os.environ.get('LLM_CLIENT_SECRET')



auth_string = f"{client_id}:{client_secret}"
auth_header = f"Basic {base64.b64encode(auth_string.encode()).decode()}"

llm_openai = ChatOpenAI()

class ApprovalState(TypedDict):
    action_details: str
    status: Optional[Literal["pending", "approved", "rejected"]]


def approval_node(state: ApprovalState) -> Command[Literal["proceed", "cancel"]]:
    # Expose details so the caller can render them in a UI
    decision = interrupt({
        "question": "Approve this action?",
        "details": state["action_details"],
    })

    # Route to the appropriate node after resume
    return Command(goto="proceed" if decision else "cancel")


def proceed_node(state: ApprovalState):
    return {"status": "approved"}


def cancel_node(state: ApprovalState):
    return {"status": "rejected"}




builder = StateGraph(ApprovalState)
builder.add_node("approval", approval_node)
builder.add_node("proceed", proceed_node)
builder.add_node("cancel", cancel_node)
builder.add_edge(START, "approval")
builder.add_edge("proceed", END)
builder.add_edge("cancel", END)

checkpoint = MemorySaver()

graph = builder.compile(checkpointer=checkpoint)

# Display Mermaid text instead of trying to render as PNG
print(graph.get_graph().draw_mermaid())

config = {"configurable": {"thread_id": "approval-456"}}
initial = graph.invoke(
    {"action_details": "Send Daily Summary", "status": "pending"},
    config=config,
)
print(initial["__interrupt__"])

---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	approval(approval)
	proceed(proceed)
	cancel(cancel)
	__end__([<p>__end__</p>]):::last
	__start__ --> approval;
	approval -.-> cancel;
	approval -.-> proceed;
	cancel --> __end__;
	proceed --> __end__;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc

[Interrupt(value={'question': 'Approve this action?', 'details': 'Send Daily Summary'}, id='749f425b9c85dd8f3966e25eb0952cf3', response_schema=None)]


In [6]:
resumed = graph.invoke(Command(resume=True), config=config)
print(resumed["status"])  # -> "rejected"


approved
